In [28]:
import json
import os
import chromadb
from typing import Annotated, TYPE_CHECKING

from IPython.display import display, HTML

from openai import AsyncOpenAI

from semantic_kernel.agents import ChatCompletionAgent, ChatHistoryAgentThread
from semantic_kernel.connectors.ai.open_ai import OpenAIChatCompletion
from semantic_kernel.contents import FunctionCallContent,FunctionResultContent, StreamingTextContent, ChatMessageContent
from semantic_kernel.contents.utils.author_role import AuthorRole
from semantic_kernel.functions import kernel_function

from semantic_kernel.connectors.ai.open_ai import OpenAIChatPromptExecutionSettings
from semantic_kernel.contents.chat_history import ChatHistory
from semantic_kernel.contents import AuthorRole

if TYPE_CHECKING:
    from chromadb.api.models.Collection import Collection
# Initialize the asynchronous OpenAI client
from dotenv import load_dotenv


In [29]:
from typing import Annotated, Dict, Any, List
from semantic_kernel.functions import kernel_function

In [30]:
import json
import os

# ------------------------------
# 动态加载 subcommand 信息
# ------------------------------
def load_subcommands(info_path="../json/modkit_subcommand_info.json", param_path="../json/modkit_subcommand_parameter.json"):
    with open(info_path, "r") as f:
        subcommand_info = json.load(f)
    with open(param_path, "r") as f:
        subcommand_parameter = json.load(f)
    return subcommand_info, subcommand_parameter

# 使用示例
SUBCOMMANDINFO, subcommand_parameter = load_subcommands()

In [31]:
import json
from typing import Dict, Any

class CodeGeneratorPlugin:

    def __init__(self):
        self.subcommand_parameter = subcommand_parameter

    @kernel_function(
        description="Get the subcommand parameter",
        name="get_subcommand_parameter"
    )
    def get_subcommand_parameter(self, subcommand_name: str) -> Dict[str, Any]:
        return self.subcommand_parameter[subcommand_name]

    @kernel_function(
        description="Generate a prompt for the LLM given tool parameters JSON and the user's task request.",
        name="generate_tool_prompt"
    )
    def generate_tool_prompt(self,tool_definition: Dict[str, Any], user_task: str) -> str:
        """
        Generate a prompt for the LLM given tool parameters JSON and the user's task request.
        English only.
        """
        formatted_params = json.dumps(tool_definition, indent=2, ensure_ascii=False)
        prompt = (
            "You are an expert systems engineer skilled in integrating command-line tools. "
            "You will be given a parameter definition and a user task. "
            "Your goal is to generate an accurate command-line invocation that fulfills the user's request.\n\n"
            f"User Task:\n{user_task}\n\n"
            "Parameter Definition:\n"
            f"{formatted_params}\n\n"
            "Instructions:\n"
            "1. Fill all required parameters using the given user task.\n"
            "2. Use relevant optional parameters if they match the user's intent.\n"
            "3. If the parameter can just be used as default, do not show it in the command."
        )
        return prompt
    


In [32]:
# ------------------------------
# 1️⃣ 初始化 OpenAI 客户端
# ------------------------------
load_dotenv()
client = AsyncOpenAI(
    api_key=os.environ["GITHUB_TOKEN"],
    base_url="https://models.inference.ai.azure.com/"
)

chat_completion_service = OpenAIChatCompletion(
    ai_model_id="gpt-4o",
    async_client=client,
)

In [33]:
from pydantic import BaseModel, ValidationError, Field

class SubTask(BaseModel):
    assigned_subcommand: str = Field(
        description="The specific subcommand assigned to handle this subtask")
    task_details: str = Field(
        description="Detailed description of what needs to be done for this subtask")


class ModkitPlan(BaseModel):
    main_task: str = Field(
        description="The overall travel request from the user")
    subtasks: List[SubTask] = Field(
        description="List of subtasks broken down from the main task, each assigned to a specialized subcommand")

In [ ]:
from semantic_kernel.functions import KernelArguments
AGENT_NAME = "ModkitAgent"

AGENT_INSTRUCTIONS = """You are an code generator agent.
    Your job is to generate the code based on the user's request and the subcommand parameter description.
    You should follow the steps:
    - Generate the code prompt based on the user's request and the subcommand parameter description.
    - Generate the code based on the code prompt in step 1.
    
    Important:
    - You should not include any other text or comments in the code.
    - Your code MUST start with 'modkit '
    - Do not show unnecessary parameters in the code but at least show the required parameters.

"""


for name, info in SUBCOMMANDINFO.items():
    AGENT_INSTRUCTIONS += ("\n - "+name+": "+info.get("description"))


# Create the prompt execution settings and configure the Pydantic model response format
settings = OpenAIChatPromptExecutionSettings(response_format=ModkitPlan)

agent = ChatCompletionAgent(
    service=chat_completion_service,
    name=AGENT_NAME,
    instructions=AGENT_INSTRUCTIONS,
    arguments=KernelArguments(settings) 
)

In [ ]:
from semantic_kernel.functions import KernelArguments
AGENT_NAME = "CodeGeneratorAgent"

AGENT_INSTRUCTIONS = """You are an code generator agent.
    Your job is to generate the code based on the user's request and the subcommand parameter description.
    You should follow the steps:
    - Generate the code prompt based on the user's request and the subcommand parameter description.
    - Generate the code based on the code prompt in step 1.
    
    Important:
    - You should not include any other text or comments in the code.
    - Your code MUST start with 'modkit '
    - Do not show unnecessary parameters in the code.
"""

code_agent = ChatCompletionAgent(
    service=chat_completion_service,
    name=AGENT_NAME,
    instructions=AGENT_INSTRUCTIONS,
    plugins=[CodeGeneratorPlugin()],
)

In [36]:
from IPython.display import display, HTML

async def main():
    # Create a thread for the agent
    # If no thread is provided, a new thread will be
    # created and returned with the initial response
    thread: ChatHistoryAgentThread | None = None

    while True:
        # Get user input interactively
        user_input = input("Enter your request (or 'exit' to quit): ")
        if user_input.strip().lower() in {"exit", "quit"}:
            print("Exiting chat...")
            break

        # Start building HTML output
        html_output = "<div style='margin-bottom:10px'>"
        html_output += "<div style='font-weight:bold'>User:</div>"
        html_output += f"<div style='margin-left:20px'>{user_input}</div>"
        html_output += "</div>"

        # Collect the agent's response
        response = await agent.get_response(messages=user_input, thread=thread)
        thread = response.thread

        try:
            # Try to validate the response as a TravelPlan
            subcommand_plan = ModkitPlan.model_validate(json.loads(response.message.content))
               
            # Display the validated model as formatted JSON
            formatted_json = subcommand_plan.model_dump_json(indent=4)

            html_output += "<div style='margin-bottom:20px'>"
            html_output += "<div style='font-weight:bold'>Modkit Subcommand Plan:</div>"
            html_output += f"<pre style='margin-left:20px; padding:10px; border-radius:5px;'>{formatted_json}</pre>"
            html_output += "</div>"
        except ValidationError as e:
            # Handle validation errors
            html_output += "<div style='margin-bottom:20px; color:red;'>"
            html_output += "<div style='font-weight:bold'>Validation Error:</div>"
            html_output += f"<pre style='margin-left:20px;'>{str(e)}</pre>"
            html_output += "</div>"
            # Add this to see what the response contains for debugging
            html_output += "<div style='margin-bottom:20px;'>"
            html_output += "<div style='font-weight:bold'>Raw Response:</div>"
            html_output += f"<div style='margin-left:20px; white-space:pre-wrap'>{response.content}</div>"
            html_output += "</div>"

        # Only attempt subtasks and code generation if travel_plan exists and has subtasks
        if 'subcommand_plan' in locals() and len(subcommand_plan.subtasks) > 0:
            for subtask in subcommand_plan.subtasks:
                subcommand_name = subtask.assigned_subcommand
                task_info = """
                Your subcommand is {subcommand_name}. 
                Your task is {subtask_details}.
                Please generate the Code.
                """
                task_info = task_info.format(subcommand_name=subcommand_name, subtask_details=subtask.task_details)
                response = await code_agent.get_response(messages=task_info, thread=thread)
                # Display the code agent's response as HTML
                html_output += "<div style='margin-bottom:20px'>"
                html_output += "<div style='font-weight:bold'>Generated Code:</div>"
                html_output += f"<pre style='margin-left:20px; padding:10px; border-radius:5px; background-color:#f6f8fa'>{response.message.content}</pre>"
                html_output += "</div>"

        html_output += "<hr>"

        # Display formatted HTML
        display(HTML(html_output))

await main()

ContentFilterAIException: <class 'semantic_kernel.connectors.ai.open_ai.services.open_ai_chat_completion.OpenAIChatCompletion'> service encountered a content error